# Bài 7: Tối ưu hiệu năng — OPTIMIZE, Z-ORDER & Small-file Problem

## Mục tiêu
- Hiểu **small-file problem**: vì sao ghi nhiều lần/streaming tạo ra rất nhiều file nhỏ và tại sao điều đó chậm.
- Dùng `OPTIMIZE` để compact (bin-pack) file nhỏ thành file lớn.
- Dùng `OPTIMIZE ... ZORDER BY` để tổ chức lại dữ liệu, tăng hiệu quả data skipping cho các cột hay filter cùng lúc.


## 7.1. Small-file problem

Mỗi lần `write`/`append`/streaming micro-batch tạo ra **file Parquet mới**. Với workload ghi thường xuyên (streaming, ingest nhiều lần/ngày), số lượng file nhỏ tăng nhanh:

- Đọc nhiều file nhỏ tốn overhead mở file, đọc footer/metadata — chậm hơn nhiều so với đọc ít file lớn cùng tổng dung lượng.
- Trên object storage (S3/MinIO), mỗi lần `list`/`open` là 1 network call — càng nhiều file càng chậm và tốn chi phí API call.
- Driver Spark phải giữ metadata (đường dẫn, thống kê) của **mọi file** trong bộ nhớ khi lập kế hoạch truy vấn.

## 7.2. `OPTIMIZE` (bin-packing / compaction)

```sql
OPTIMIZE db.tbl;                          -- compact toan bang
OPTIMIZE db.tbl WHERE order_date = '2024-01-01';   -- chi 1 partition (giam chi phi)
```

`OPTIMIZE` đọc các file nhỏ, gộp lại thành các file có kích thước mục tiêu (mặc định ~1GB, cấu hình qua `spark.databricks.delta.optimize.maxFileSize` hoặc bảng property `delta.targetFileSize`), rồi commit `remove` file cũ + `add` file mới — **không đổi dữ liệu**, chỉ đổi cách file được tổ chức. Các version cũ vẫn time-travel được bình thường (file cũ chỉ mất khi VACUUM).

Nên chạy `OPTIMIZE` định kỳ (không phải sau mỗi lần ghi) — ví dụ 1 lần/ngày cho bảng ingest liên tục, để cân bằng chi phí compaction với lợi ích đọc.

## 7.3. `ZORDER BY` — data clustering đa chiều

```sql
OPTIMIZE db.tbl ZORDER BY (customer_id, event_date);
```

Z-Ordering sắp xếp lại dữ liệu trong các file sao cho các giá trị **gần nhau theo nhiều cột cùng lúc** được lưu gần nhau vật lý (dùng đường cong Z-order/Morton code). Kết quả: thống kê min/max mỗi file "chặt" hơn nhiều theo các cột đó → **data skipping hiệu quả hơn hẳn** khi query filter theo các cột đã Z-order, đặc biệt hữu ích cho cột **không phải partition column** hoặc khi cần filter hiệu quả theo **nhiều cột cùng lúc** (partition chỉ tối ưu tốt cho 1-2 cột cardinality thấp).

Chọn cột Z-order: ưu tiên cột **hay xuất hiện trong WHERE** và có **cardinality cao** (ngược lại với chọn cột partition) — ví dụ `customer_id`, `device_id`. Không nên Z-order theo quá nhiều cột (thường ≤ 3-4 cột) vì lợi ích giảm dần.

## 7.4. `ANALYZE TABLE` — cập nhật thống kê

```sql
ANALYZE TABLE db.tbl COMPUTE STATISTICS FOR COLUMNS col1, col2;
```
Cập nhật statistics dùng cho cost-based optimizer (CBO) chọn join strategy phù hợp — khác với min/max stats tự động của Delta (dùng cho data skipping), đây là thống kê tổng hợp toàn bảng (số dòng, NDV, histogram...).


## 0. Thiết lập môi trường

Notebook này chạy trong container `spark-master` (Jupyter Lab, xem `startup.sh`), kết nối tới:
- **Spark cluster**: `spark://spark-master:7077`
- **Hive Metastore**: `thrift://hive-metastore:9083` (dùng làm catalog)
- **MinIO** (S3-compatible): dữ liệu bảng managed nằm dưới `s3a://data-platform/managed/...`

Image `deltaio/delta-docker` đã cấu hình sẵn Delta Lake trong `spark-defaults.conf` nên không cần khai báo `spark.jars.packages` mỗi lần tạo `SparkSession`.

Yêu cầu: `docker compose up -d hive-metastore minio spark-master spark-worker` đã chạy trước khi mở notebook này.


In [ ]:
from pyspark.sql import SparkSession

DB_NAME = "bai07"
WAREHOUSE_DIR = "s3a://data-platform/managed"

spark = (
    SparkSession.builder
    .appName("bai07-optimize")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-master")
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR)
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME} LOCATION '{WAREHOUSE_DIR}/{DB_NAME}.db'")
spark.sql(f"USE {DB_NAME}")
spark


## 7.5. Ví dụ minh hoạ

In [ ]:
import random
spark.sql("DROP TABLE IF EXISTS bai07.events")

# Mo phong nhieu lan ghi nho (giong streaming/ingest thuong xuyen) -> tao nhieu file nho
for i in range(8):
    batch = spark.createDataFrame(
        [(i * 10 + j, random.choice(["A","B","C"]), random.randint(1, 1000)) for j in range(5)],
        ["event_id", "customer_id", "amount"],
    )
    batch.write.format("delta").mode("append").saveAsTable("bai07.events")

loc = spark.sql("DESCRIBE EXTENDED bai07.events").filter("col_name = 'Location'").collect()[0]["data_type"]
p = sc._jvm.org.apache.hadoop.fs.Path(loc)
fs3 = p.getFileSystem(sc._jsc.hadoopConfiguration())
n_before = len([f for f in fs3.listStatus(p) if f.getPath().getName().endswith(".parquet")])
print("So file Parquet TRUOC OPTIMIZE:", n_before)


In [ ]:
result = spark.sql("OPTIMIZE bai07.events")
result.show(truncate=False)

n_after = len([f for f in fs3.listStatus(p) if f.getPath().getName().endswith(".parquet")])
print("So file Parquet SAU OPTIMIZE (file cu van con tren storage, chi khong con 'active'):", n_after)

# So file dang active theo Delta log (dung DeltaTable de kiem tra qua history)
spark.sql("DESCRIBE HISTORY bai07.events").select("version","operation","operationMetrics").show(truncate=False)


In [ ]:
# ZORDER: to chuc lai file theo customer_id de cac gia tri gan nhau nam cung file
spark.sql("OPTIMIZE bai07.events ZORDER BY (customer_id)").show(truncate=False)

spark.sql("SELECT * FROM bai07.events WHERE customer_id = 'B'").explain(True)


## 7.6. Thực hành

**Bài 1** — Tạo bảng `bai07.clicks (click_id INT, user_id STRING, page STRING)`. Ghi dữ liệu bằng **10 lần append riêng lẻ** (mỗi lần vài dòng) để mô phỏng nhiều file nhỏ.

**Bài 2** — Đếm số file Parquet vật lý trước khi `OPTIMIZE`. Chạy `OPTIMIZE bai07.clicks` rồi xem kết quả trả về (`metrics` — số file trước/sau).

**Bài 3** — Chạy `OPTIMIZE bai07.clicks ZORDER BY (user_id)`. So sánh `.explain(True)` của 1 query `WHERE user_id = '...'` trước và sau khi Z-order (chú ý phần thống kê/scan).

**Bài 4** — Nếu bảng có cột partition là `page` và bạn muốn tối ưu chỉ 1 giá trị `page` cụ thể (không compact cả bảng), viết câu lệnh `OPTIMIZE` phù hợp.

**Bài 5 (tư duy)** — Bảng `bai07.clicks` có 2 cột hay được filter cùng lúc: `user_id` (cardinality rất cao) và `page` (cardinality thấp, chỉ ~10 giá trị). Bạn sẽ chọn `page` làm **partition column**, `user_id` làm **Z-order column**, hay ngược lại? Giải thích.


### Vùng làm bài — Bài 1

In [ ]:
# TODO: Bài 1


### Vùng làm bài — Bài 2

In [ ]:
# TODO: Bài 2


### Vùng làm bài — Bài 3

In [ ]:
# TODO: Bài 3


### Vùng làm bài — Bài 4

_Viết câu lệnh SQL của bạn ở đây._

### Vùng làm bài — Bài 5

_Viết câu trả lời của bạn ở đây._

---
## Gợi ý / đáp án tham khảo

In [ ]:
# Dap an Bai 1
spark.sql("DROP TABLE IF EXISTS bai07.clicks")
pages = ["home", "cart", "checkout"]
for i in range(10):
    b = spark.createDataFrame(
        [(i*3+j, f"user_{random.randint(1,500)}", random.choice(pages)) for j in range(3)],
        ["click_id", "user_id", "page"],
    )
    b.write.format("delta").mode("append").saveAsTable("bai07.clicks")


In [ ]:
# Dap an Bai 2
loc2 = spark.sql("DESCRIBE EXTENDED bai07.clicks").filter("col_name = 'Location'").collect()[0]["data_type"]
p2 = sc._jvm.org.apache.hadoop.fs.Path(loc2)
fs4 = p2.getFileSystem(sc._jsc.hadoopConfiguration())
print("Truoc:", len([f for f in fs4.listStatus(p2) if f.getPath().getName().endswith(".parquet")]))

res = spark.sql("OPTIMIZE bai07.clicks")
res.show(truncate=False)


In [ ]:
# Dap an Bai 3
print("== Truoc Z-ORDER (query da OPTIMIZE binh thuong) ==")
spark.sql("SELECT * FROM bai07.clicks WHERE user_id = 'user_1'").explain(True)

spark.sql("OPTIMIZE bai07.clicks ZORDER BY (user_id)").show(truncate=False)

print("== Sau Z-ORDER ==")
spark.sql("SELECT * FROM bai07.clicks WHERE user_id = 'user_1'").explain(True)


**Đáp án Bài 4**:
```sql
OPTIMIZE bai07.clicks WHERE page = 'checkout';
```
Chỉ compact các file thuộc partition `page = 'checkout'`, tiết kiệm chi phí so với compact toàn bảng.

**Đáp án Bài 5**: `page` (cardinality thấp, ~10 giá trị) hợp làm **partition column** — tạo ra số lượng thư mục hợp lý, hỗ trợ partition pruning gọn gàng. `user_id` (cardinality rất cao — hàng trăm nghìn/triệu user) **không nên partition** (sẽ tạo hàng triệu thư mục nhỏ → chính là small-file problem), nhưng rất hợp để **Z-order**, vì Z-order không tạo thêm thư mục, chỉ sắp xếp lại vị trí dòng bên trong các file hiện có để thống kê min/max theo `user_id` "chặt" hơn, giúp data skipping hiệu quả mà không phá vỡ cấu trúc partition.
